In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Inputs

In [2]:
features_ranked_ordered = pd.read_parquet(variables.FEATURES_RANKED_ORDERED).iloc[:variables.FEATURES_UNIQUE_RANKED_LIMIT]
features = pd.read_parquet(variables.FEATURES_DATASET_FOR_SELECTION_PATH)

Subset sample

In [3]:
# Randomly sample index
seed = np.random.default_rng(42)
index = seed.choice(len(features), size=variables.FEATURE_SELECTION_SUBSAMPLE_AMOUNT, replace=False)
index.sort()

features = features.iloc[index]

Core logic 

Iterates over n number of features

In [4]:
import warnings

def compute_correlations(item):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pearson = features.corrwith(features[item], method="pearson").abs().drop(item).fillna(0)
        spearman = features.corrwith(features[item], method="spearman").abs().drop(item).fillna(0)
    return pearson, spearman

corr_results = run_parallel(function=compute_correlations, data=features_ranked_ordered.index.tolist())


os=windows cpu_count=8 max_assigned_workers=1


Processing..:   0%|          | 0/100 [00:00<?, ?item/s]

Processing..: 100%|██████████| 100/100 [00:58<00:00,  1.72item/s]


In [5]:
pd.DataFrame(corr_results)[:3]

,0,1
0,nsw_price_asinh_lag_2 0.94...,nsw_price_asinh_lag_2 0.93...
1,nsw_price_asinh_lag_1 0.94...,nsw_price_asinh_lag_1 0.93...
2,nsw_price_asinh_lag_1 0.90...,nsw_price_asinh_lag_1 0.90...


Clean up

In [6]:
LINEAR_CORR_THRESHOLD = 0.95
NONLINEAR_CORR_THRESHOLD = 0.95

unique_features = []
duplicate_features = []

for i, feature in enumerate(features_ranked_ordered.index):
    pearson_corr, spearman_corr = corr_results[i]

    is_linear_duplicate = (pearson_corr[unique_features] > LINEAR_CORR_THRESHOLD).any()
    is_nonlinear_duplicate = (spearman_corr[unique_features] > NONLINEAR_CORR_THRESHOLD).any()
    is_duplicate = bool(is_linear_duplicate or is_nonlinear_duplicate)


    if is_duplicate:
        duplicate_features.append(feature)
    else:
        unique_features.append(feature)
        
    del pearson_corr, spearman_corr

del corr_results


In [7]:
features_ranked_ordered_duplicate = features_ranked_ordered.loc[duplicate_features].reset_index().rename(columns={"index": "feature"})
features_ranked_ordered_unique = features_ranked_ordered.loc[unique_features].reset_index().rename(columns={"index": "feature"})
features_ranked_ordered_unique.index = features_ranked_ordered_unique['feature']
features_ranked_ordered_unique = features_ranked_ordered_unique.drop(columns='feature')

In [8]:
features.shape

(200, 634)

In [9]:
features_unique_data = features[unique_features]

In [10]:
features_unique_data.shape

(200, 71)

In [11]:
features_ranked_ordered_unique.shape

(71, 96)

View

In [12]:
display(features_ranked_ordered_unique[:10])
display(features_ranked_ordered_duplicate[:10])
print(len(features_ranked_ordered_duplicate))

,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,target_h11,target_h12,target_h13,target_h14,target_h15,target_h16,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,target_h25,target_h26,target_h27,target_h28,target_h29,target_h30,target_h31,target_h32,target_h33,target_h34,target_h35,target_h36,target_h37,target_h38,target_h39,target_h40,target_h41,target_h42,target_h43,target_h44,target_h45,target_h46,target_h47,target_h48,target_h49,target_h50,target_h51,target_h52,target_h53,target_h54,target_h55,target_h56,target_h57,target_h58,target_h59,target_h60,target_h61,target_h62,target_h63,target_h64,target_h65,target_h66,target_h67,target_h68,target_h69,target_h70,target_h71,target_h72,target_h73,target_h74,target_h75,target_h76,target_h77,target_h78,target_h79,target_h80,target_h81,target_h82,target_h83,target_h84,target_h85,target_h86,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
feature,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
nsw_price_asinh_lag_1,4,4,3,5,8,1,1,5,3,1,3,2,4,1,1,1,3,2,3,3,3,1,2,1,3,7,1,8,2,2,2,1,8,15,2,1,1,4,2,3,13,15,8,15,6,5,16,8,15,11,18,2,3,12,32,17,33,26,26,46,5,8,9,34,9,17,50,1,4,42,30,60,38,26,27,19,61,30,126,34,117,64,148,29,76,42,91,92,15,50,17,34,31,67,30,58
nsw_price_asinh_lag_2,11,9,7,9,5,7,7,9,10,3,10,7,7,8,6,4,7,7,5,4,6,13,3,3,4,3,15,2,14,11,17,11,12,16,17,19,18,12,21,12,5,34,15,12,16,20,23,10,17,20,67,44,5,11,17,3,14,34,38,7,2,3,5,24,10,23,30,33,36,63,92,56,129,92,51,44,66,38,86,99,52,46,94,26,39,141,58,38,40,82,108,51,38,167,86,131
nsw_price_asinh_lag_4,13,13,12,13,13,12,13,12,11,8,12,19,20,14,13,12,17,25,16,21,18,3,18,18,16,16,11,6,4,12,9,16,1,11,9,6,14,5,8,15,4,16,2,10,8,16,4,13,9,14,6,27,7,8,18,7,10,24,85,6,25,22,17,46,37,5,47,44,32,71,68,12,13,28,18,16,14,20,61,106,16,55,76,25,91,57,74,30,24,23,30,16,43,32,45,40
nsw_price_asinh_lag_12,30,30,26,31,21,30,29,21,18,27,24,27,21,28,26,16,14,14,13,11,9,27,11,14,7,17,9,18,22,6,20,21,19,21,12,30,30,34,17,27,18,11,11,17,33,44,41,22,69,80,47,46,152,47,71,52,18,4,44,23,10,54,27,35,27,55,79,35,50,82,91,61,24,33,14,34,125,52,245,53,105,125,116,41,42,31,22,42,46,13,55,60,47,22,14,27
nsw_price_asinh_lag_48,101,125,87,114,57,101,130,103,109,127,82,98,75,110,81,59,87,80,73,93,64,100,135,61,52,71,80,102,83,134,110,78,104,99,107,135,77,56,41,175,48,54,85,122,58,36,17,29,112,143,104,53,77,22,6,68,35,53,4,26,40,46,32,69,41,48,35,25,69,78,33,14,23,29,9,53,19,25,24,111,36,15,14,17,16,9,9,12,19,36,19,24,27,15,10,10
nsw_price_asinh_lag_96,317,236,216,258,244,303,195,158,198,223,153,163,155,179,298,247,203,148,159,266,209,264,212,287,178,154,169,127,104,159,216,149,143,137,158,70,52,103,23,131,116,113,69,159,134,243,265,288,209,248,273,352,124,202,141,141,194,187,187,100,201,168,197,142,149,137,126,46,178,99,276,30,70,57,131,78,30,118,28,119,133,143,32,74,43,81,75,57,160,130,80,71,71,229,113,166
nsw_price_asinh_lag_336,200,278,170,173,175,182,182,223,168,237,188,165,134,219,207,97,271,254,185,185,166,156,182,230,233,138,240,202,204,208,186,258,216,321,323,167,132,346,111,298,218,207,292,338,222,253,324,225,272,233,257,172,269,386,310,241,146,200,312,312,305,276,251,107,135,144,171,106,92,16,29,119,183,160,83,126,110,238,72,159,35,120,43,192,163,313,96,64,216,232,200,275,212,307,350,314
nsw_price_asinh_lag_337,217,123,144,201,211,209,155,147,219,195,239,151,159,156,224,134,178,181,141,124,140,168,166,212,137,173,201,114,283,328,311,259,274,266,385,189,110,221,169,180,164,342,241,220,135,246,296,138,219,356,278,176,362,264,245,190,283,256,228,319,318,218,181,171,84,45,314,142,212,69,195,215,187,262,219,290,233,224,244,233,155,117,98,174,272,272,70,225,252,193,183,194,244,290,278,293
nsw_price_asinh_rmean_48,37,29,18,26,23,27,27,24,29,19,17,21,29,29,34,29,22,31,15,27,16,11,5,7,19,22,12,19,38,24,43,19,28,24,34,29,26,29,32,19,28,14,27,21,15,9,11,20,18,28,13,14,22,23

,feature,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,target_h11,target_h12,target_h13,target_h14,target_h15,target_h16,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,target_h25,target_h26,target_h27,target_h28,target_h29,target_h30,target_h31,target_h32,target_h33,target_h34,target_h35,target_h36,target_h37,target_h38,target_h39,target_h40,target_h41,target_h42,target_h43,target_h44,target_h45,target_h46,target_h47,target_h48,target_h49,target_h50,target_h51,target_h52,target_h53,target_h54,target_h55,target_h56,target_h57,target_h58,target_h59,target_h60,target_h61,target_h62,target_h63,target_h64,target_h65,target_h66,target_h67,target_h68,target_h69,target_h70,target_h71,target_h72,target_h73,target_h74,target_h75,target_h76,target_h77,target_h78,target_h79,target_h80,target_h81,target_h82,target_h83,target_h84,target_h85,target_h86,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
0,nsw_price_asinh_lag_335,193,176,148,179,116,151,132,159,150,172,134,105,89,175,138,70,155,184,95,157,205,144,123,130,141,136,146,101,146,173,159,171,149,280,172,86,48,109,27,174,203,191,210,180,115,178,207,56,80,124,53,100,244,114,223,75,122,102,251,131,184,240,132,114,161,56,125,72,166,11,90,111,159,157,147,86,98,63,46,192,40,76,27,35,86,39,115,58,76,47,93,84,169,91,140,97
1,qld_price_asinh_lag_2,26,23,23,27,22,23,19,18,24,25,20,13,16,19,10,14,19,13,23,25,23,9,16,22,23,30,33,20,24,20,36,31,22,20,23,22,22,26,12,23,38,66,26,42,53,28,14,67,32,82,134,94,115,38,99,41,52,73,60,80,8,36,11,147,81,31,153,39,156,145,77,59,180,128,200,123,191,110,168,152,271,254,163,109,84,97,95,73,248,54,105,95,84,185,204,102
2,qld_price_asinh_lag_335,316,229,205,149,166,192,193,219,177,211,185,113,83,80,232,209,226,154,115,160,163,106,247,144,153,168,257,150,192,193,130,240,178,138,204,92,62,233,110,101,172,176,302,265,182,183,209,185,224,135,74,75,179,56,248,89,190,121,94,133,245,195,75,78,106,72,130,94,195,97,75,48,48,70,64,43,49,71,45,23,79,167,93,106,195,232,88,146,140,163,139,120,114,144,187,170
3,qld_price_asinh_rmean_336,80,99,99,121,97,92,80,100,106,85,83,70,59,48,72,77,59,51,53,67,33,65,43,91,84,97,61,112,78,91,63,55,70,80,96,97,113,76,75,59,73,59,59,58,31,79,28,31,22,52,120,31,50,59,68,37,5,20,6,40,65,29,14,27,17,52,41,21,61,65,2,35,5,10,57,97,34,27,15,40,32,19,15,30,36,47,29,55,34,21,131,132,80,106,132,59
4,sa_price_asinh_lag_335,146,207,294,374,284,247,254,332,294,240,361,270,222,408,326,307,310,271,228,343,220,363,274,313,334,335,326,424,282,407,352,313,278,423,432,310,304,300,329,268,374,340,290,309,327,323,332,328,239,205,286,319,390,279,148,208,203,199,264,229,267,413,285,368,314,189,243,187,189,359,235,351,339,335,309,331,416,164,339,332,256,346,333,259,256,343,193,188,133,215,342,412,215,382,251,245
5,sa_price_asinh_lag_337,281,253,281,230,291,194,250,361,314,424,362,277,214,234,337,351,268,258,229,349,233,314,310,258,328,230,308,294,350,296,264,304,271,340,305,258,290,285,354,284,380,233,205,292,367,369,315,153,166,173,268,125,219,322,257,229,291,310,354,276,340,313,210,186,223,196,260,192,139,324,152,334,380,439,173,289,256,90,248,165,199,287,336,306,384,139,218,193,206,245,168,145,421,321,279,290
6,vic_price_asinh_lag_335,263,158,256,202,205,229,242,226,250,239,222,245,185,228,338,284,177,216,240,372,261,400,266,340,267,211,330,354,259,199,339,342,312,393,406,266,208,274,242,299,251,309,256,224,117,216,142,166,111,168,208,270,161,166,315,136,324,162,118,249,249,297,155,159,245,110,224,234,234,258,247,196,290,291,272,239,306,133,261,361,289,296,265,258,242,208,178,179,278,252,214,345,254,286,205,219
7,vic_price_asinh_lag_337,273,289,233,291,260,333,286,248,253,307,330,297,237,240,277,322,222,305,214,334,342,230,308,395,274,324,370,425,405,356,337,377,371,367,439,268,269,487,299,283,409,443,319,330,262,387,308,341,248,366,307,272,347,380,337,315,377,343,276,368,449,278,248,177,362,186,280,270,29

29


Export

In [ ]:
features_ranked_ordered_duplicate.to_csv("features_ranked_ordered_duplicate")

In [13]:
features_ranked_ordered_unique.to_parquet(variables.FEATURES_RANKED_ORDERED_UNIQUE_PATH)
features_unique_data.to_parquet(variables.FEATURES_UNIQUE_DATA_PATH)